# Previsão dos Fatores de Satisfação dos Funcionários
### Pipeline Completo de Machine Learning

**Dataset:** IBM HR Analytics Employee Attrition & Performance  
**Objetivo:** Identificar os fatores que influenciam a satisfação e construir um modelo preditivo robusto  
**Variável Alvo:** `JobSatisfaction` (escala ordinal 1–4)  
**Tipo de Problema:** Classificação Multiclasse  

---

| Etapa | Descrição |
|-------|-----------|
| 1 | Entendimento do Problema |
| 2 | Importação e Exploração dos Dados (EDA) |
| 3 | Limpeza e Tratamento dos Dados |
| 4 | Engenharia de Atributos |
| 5 | Encoding e Escalonamento |
| 6 | Seleção de Features |
| 7 | Divisão Treino / Validação / Teste |
| 8 | Treinamento de Múltiplos Modelos |
| 9 | Otimização de Hiperparâmetros |
| 10 | Avaliação e Interpretabilidade |
| 11 | Persistência do Modelo |
| 12 | Insights e Recomendações de Negócio |


## 0. Setup e Importações

In [1]:
import os, sys, warnings, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import joblib

warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold,
    RandomizedSearchCV, GridSearchCV
)
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler,
    LabelEncoder, label_binarize
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.feature_selection import (
    SelectKBest, mutual_info_classif, f_classif
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
    roc_curve, auc, precision_recall_curve
)

# Gradient Boosting avançado (instalar se necessário: pip install xgboost lightgbm catboost)
try:
    from xgboost import XGBClassifier
    print("✓ XGBoost disponível")
except ImportError:
    print("✗ XGBoost não instalado")

try:
    from lightgbm import LGBMClassifier
    print("✓ LightGBM disponível")
except ImportError:
    print("✗ LightGBM não instalado")

try:
    from catboost import CatBoostClassifier
    print("✓ CatBoost disponível")
except ImportError:
    print("✗ CatBoost não instalado")

# SHAP
try:
    import shap
    print("✓ SHAP disponível")
except ImportError:
    print("✗ SHAP não instalado (pip install shap)")

# Configuração visual
sns.set_theme(style='whitegrid', palette='viridis')
plt.rcParams.update({'font.size': 11, 'axes.titlesize': 13, 'figure.dpi': 120})
SAT_COLORS = ['#d32f2f', '#f57c00', '#388e3c', '#1976d2']

print("\n✓ Todas as importações realizadas com sucesso!")
print(f"  Pandas:      {pd.__version__}")
print(f"  NumPy:       {np.__version__}")


ModuleNotFoundError: No module named 'pandas'

## 1. Entendimento do Problema

### 1.1 Contexto de Negócio

A **satisfação dos funcionários** é um indicador crítico de saúde organizacional. Empresas com alta satisfação no trabalho apresentam:
- Menor taxa de rotatividade (attrition)
- Maior produtividade e qualidade
- Melhor clima organizacional
- Redução de custos com recrutamento e treinamento

### 1.2 Variável Alvo

**`JobSatisfaction`** — escala ordinal de 1 a 4:
- `1` = Baixa satisfação
- `2` = Média-Baixa satisfação  
- `3` = Média-Alta satisfação
- `4` = Alta satisfação

### 1.3 Métricas de Sucesso

| Métrica | Justificativa |
|---------|---------------|
| **F1-Weighted** | Lida com desbalanceamento entre classes |
| **F1-Macro** | Penaliza igualmente erros em todas as classes |
| **Accuracy** | Baseline geral |
| **ROC-AUC** | Capacidade discriminativa por classe |
| **Confusion Matrix** | Análise de erros por classe |

> 📌 **Meta:** Superar baseline naive (classe majoritária) com F1-Weighted ≥ 0.30


In [ ]:
# Configurar caminhos
BASE_DIR = os.path.abspath('.')
DATA_RAW       = os.path.join(BASE_DIR, 'data', 'raw', 'Employee-Attrition.csv')
DATA_PROCESSED = os.path.join(BASE_DIR, 'data', 'processed')
FIGURES_DIR    = os.path.join(BASE_DIR, 'reports', 'figures')
MODELS_DIR     = os.path.join(BASE_DIR, 'models')
REPORTS_DIR    = os.path.join(BASE_DIR, 'reports')

for d in [DATA_PROCESSED, FIGURES_DIR, MODELS_DIR, REPORTS_DIR]:
    os.makedirs(d, exist_ok=True)

TARGET = 'JobSatisfaction'
print(f"Base directory: {BASE_DIR}")
print("✓ Diretórios configurados")


## 2. Importação e Análise Exploratória dos Dados (EDA)

In [ ]:
# Carregar dados
df = pd.read_csv(DATA_RAW)
print(f"✓ Dataset carregado: {df.shape[0]:,} linhas × {df.shape[1]} colunas")
df.head()


In [ ]:
# Dimensões e tipos de dados
print("=" * 55)
print("INFORMAÇÕES DO DATASET")
print("=" * 55)
print(f"Linhas:    {df.shape[0]:,}")
print(f"Colunas:   {df.shape[1]}")
print(f"Duplicatas: {df.duplicated().sum()}")
print(f"Nulos totais: {df.isnull().sum().sum()}")
print()
print("Tipos de dados:")
print(df.dtypes.value_counts().to_string())


In [ ]:
# Estatísticas descritivas
df.describe().round(2)


In [ ]:
# Valores nulos por coluna
null_counts = df.isnull().sum()
print("Valores Nulos por Coluna:")
print(null_counts[null_counts > 0].to_string() if null_counts.sum() > 0 else "  Nenhum valor nulo encontrado! ✓")


In [ ]:
# Distribuição da variável alvo
print("Distribuição de JobSatisfaction:")
for nivel, count in df[TARGET].value_counts().sort_index().items():
    pct = count / len(df) * 100
    bar = '█' * int(pct / 2)
    label = {1:'Baixa', 2:'Média-Baixa', 3:'Média-Alta', 4:'Alta'}.get(nivel, str(nivel))
    print(f"  Nível {nivel} ({label:<12}): {count:4d} ({pct:.1f}%) {bar}")

print(f"\nMédia: {df[TARGET].mean():.2f} | Mediana: {df[TARGET].median():.1f} | Std: {df[TARGET].std():.2f}")


In [ ]:
# Visualização da distribuição do target
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
counts = df[TARGET].value_counts().sort_index()
labels = {1:'1-Baixa', 2:'2-Média-Baixa', 3:'3-Média-Alta', 4:'4-Alta'}
xlabs = [labels[i] for i in counts.index]

axes[0].bar(xlabs, counts.values, color=SAT_COLORS, edgecolor='white', linewidth=1.5)
axes[0].set_title('Contagem por Nível de Satisfação')
axes[0].set_ylabel('Nº de Funcionários')
for i, v in enumerate(counts.values):
    axes[0].text(i, v+5, str(v), ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=xlabs, autopct='%1.1f%%', colors=SAT_COLORS,
            startangle=90, wedgeprops={'edgecolor':'white','linewidth':1.5})
axes[1].set_title('Proporção por Nível de Satisfação')

fig.suptitle('Distribuição da Satisfação dos Funcionários', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Colunas categóricas — análise de categorias únicas
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f"Colunas categóricas ({len(cat_cols)}):")
for col in cat_cols:
    print(f"  {col:<30} {df[col].nunique()} categorias: {df[col].unique()[:5].tolist()}")


In [ ]:
# Identificar colunas sem valor preditivo (constantes)
const_cols = [col for col in df.columns if df[col].nunique() == 1]
print(f"Colunas constantes (sem valor preditivo): {const_cols}")

low_var = [col for col in df.columns if df[col].nunique() <= 2 and col not in ['Attrition','OverTime']]
print(f"Colunas de baixa variância: {low_var}")


In [ ]:
# Heatmap de correlação
fig, ax = plt.subplots(figsize=(16, 12))
num_df = df.select_dtypes(include=[np.number])
corr = num_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0,
            linewidths=0.3, ax=ax, vmin=-1, vmax=1, annot=False)
ax.set_title('Heatmap de Correlação entre Variáveis Numéricas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Correlação com target
print("\nTop 10 variáveis mais correlacionadas com JobSatisfaction:")
corr_target = num_df.corr()[TARGET].drop(TARGET).abs().sort_values(ascending=False).head(10)
for feat, val in corr_target.items():
    direction = '+' if num_df.corr()[TARGET][feat] > 0 else '-'
    print(f"  {direction} {feat:<40} |r| = {val:.4f}")


In [ ]:
# Análise por variáveis categóricas relevantes
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
categorical_analysis = ['Department', 'JobRole', 'OverTime', 'MaritalStatus', 'BusinessTravel', 'Gender']

for i, col in enumerate(categorical_analysis):
    means = df.groupby(col)[TARGET].mean().sort_values()
    axes.flatten()[i].barh(means.index, means.values,
                           color=sns.color_palette('viridis', len(means)), alpha=0.85)
    axes.flatten()[i].set_title(f'Satisfação por {col}', fontsize=11)
    axes.flatten()[i].set_xlabel('Satisfação Média')
    for bar, val in zip(axes.flatten()[i].patches, means.values):
        axes.flatten()[i].text(bar.get_width()+0.05, bar.get_y()+bar.get_height()/2,
                               f'{val:.2f}', va='center', fontsize=9)

fig.suptitle('Satisfação Média por Variáveis Categóricas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Análise de outliers — IQR Method
print("ANÁLISE DE OUTLIERS (Método IQR):")
print("-" * 50)
num_cols_analysis = ['Age', 'MonthlyIncome', 'TotalWorkingYears', 'YearsAtCompany',
                     'DistanceFromHome', 'DailyRate', 'HourlyRate', 'MonthlyRate']

for col in num_cols_analysis:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
    print(f"  {col:<30}: {outliers:3d} outliers ({outliers/len(df)*100:.1f}%)")


## 3. Limpeza e Tratamento dos Dados

In [ ]:
# Limpeza dos dados
df_clean = df.copy()

# 1. Remover duplicatas
print(f"Duplicatas removidas: {df_clean.duplicated().sum()}")
df_clean = df_clean.drop_duplicates()

# 2. Remover colunas sem valor preditivo
cols_drop = ['EmployeeCount', 'StandardHours', 'Over18', 'EmployeeNumber']
df_clean = df_clean.drop(columns=cols_drop)
print(f"Colunas removidas: {cols_drop}")

# 3. Converter variáveis binárias para numérico
df_clean['Attrition'] = df_clean['Attrition'].map({'Yes': 1, 'No': 0})
df_clean['OverTime']   = df_clean['OverTime'].map({'Yes': 1, 'No': 0})
print("Attrition e OverTime convertidos para binário (0/1)")

print(f"\n✓ Shape após limpeza: {df_clean.shape}")
df_clean.head()


## 4. Engenharia de Atributos (Feature Engineering)

Criamos novas variáveis baseadas no conhecimento do domínio de RH:

| Feature | Lógica | Justificativa |
|---------|--------|---------------|
| `SatisfacaoComposta` | Média de 4 dimensões de satisfação | Captura satisfação multidimensional |
| `EstabilidadeEmpresa` | YearsAtCompany / TotalWorkingYears | Indica comprometimento com a empresa |
| `Senioridade` | JobLevel × YearsAtCompany | Proxy de posição hierárquica real |
| `PressaoTrabalho` | OverTime + DistanceFromHome/max | Pressão combinada sobre o funcionário |
| `TempoSemPromocaoRelativo` | YearsSinceLastPromotion / YearsAtCompany | Percepção de estagnação na carreira |
| `FaixaEtaria` | Grupos etários ordinais (0-3) | Captura diferenças geracionais |
| `FaixaRenda` | Quartis de renda mensal | Posição relativa na distribuição salarial |


In [ ]:
# Feature Engineering
df_feat = df_clean.copy()

# 1. Satisfação Composta
sat_cols = ['EnvironmentSatisfaction', 'RelationshipSatisfaction',
            'WorkLifeBalance', 'JobInvolvement']
df_feat['SatisfacaoComposta'] = df_feat[sat_cols].mean(axis=1).round(3)

# 2. Estabilidade na empresa
df_feat['EstabilidadeEmpresa'] = np.where(
    df_feat['TotalWorkingYears'] > 0,
    df_feat['YearsAtCompany'] / (df_feat['TotalWorkingYears'] + 1),
    0
).round(3)

# 3. Senioridade
df_feat['Senioridade'] = df_feat['JobLevel'] * df_feat['YearsAtCompany']

# 4. Pressão de trabalho
df_feat['PressaoTrabalho'] = (
    df_feat['OverTime'] +
    df_feat['DistanceFromHome'] / df_feat['DistanceFromHome'].max()
).round(3)

# 5. Tempo sem promoção relativo
df_feat['TempoSemPromocaoRelativo'] = np.where(
    df_feat['YearsAtCompany'] > 0,
    df_feat['YearsSinceLastPromotion'] / (df_feat['YearsAtCompany'] + 1),
    0
).round(3)

# 6. Faixa etária ordinal
df_feat['FaixaEtaria'] = pd.cut(
    df_feat['Age'], bins=[17, 25, 35, 45, 60], labels=[0, 1, 2, 3]
).astype(int)

# 7. Faixa de renda (quartis)
df_feat['FaixaRenda'] = pd.qcut(
    df_feat['MonthlyIncome'], q=4, labels=[0, 1, 2, 3]
).astype(int)

new_features = ['SatisfacaoComposta', 'EstabilidadeEmpresa', 'Senioridade',
                'PressaoTrabalho', 'TempoSemPromocaoRelativo', 'FaixaEtaria', 'FaixaRenda']

print(f"✓ {len(new_features)} novas features criadas!")
print(f"Shape: {df_clean.shape} → {df_feat.shape}")
print("\nEstatísticas das novas features:")
df_feat[new_features].describe().round(3)


## 5. Encoding de Variáveis Categóricas

In [ ]:
# Encoding — One-Hot Encoding para variáveis nominais
# Justificativa: variáveis sem ordem natural → OHE evita relação ordinal implícita

ohe_cols = ['BusinessTravel', 'Department', 'EducationField', 'JobRole', 'MaritalStatus', 'Gender']
print("One-Hot Encoding aplicado em:")
for col in ohe_cols:
    print(f"  {col}: {df_feat[col].unique().tolist()}")

df_encoded = pd.get_dummies(df_feat, columns=ohe_cols, drop_first=False, dtype=int)
print(f"\nShape antes do encoding: {df_feat.shape}")
print(f"Shape após encoding:     {df_encoded.shape}")
print(f"Novas colunas OHE: {df_encoded.shape[1] - df_feat.shape[1]}")


## 6. Seleção de Features e Divisão dos Dados

In [ ]:
# Preparar X e y
X = df_encoded.drop(columns=[TARGET]).select_dtypes(include=[np.number]).fillna(0)
y = df_encoded[TARGET]

print(f"Features disponíveis: {X.shape[1]}")
print(f"Amostras:             {X.shape[0]}")
print(f"\nDistribuição do target:")
for cls, cnt in y.value_counts().sort_index().items():
    print(f"  Classe {cls}: {cnt} ({cnt/len(y)*100:.1f}%)")


In [ ]:
# Divisão: 70% Treino | 15% Validação | 15% Teste
# Estratificada para manter proporção das classes

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

print(f"Treino:     {X_train.shape[0]} amostras ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Validação:  {X_val.shape[0]} amostras ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"Teste:      {X_test.shape[0]} amostras ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"\n✓ Estratificação verificada:")
for split_name, y_split in [('Treino',y_train),('Val',y_val),('Teste',y_test)]:
    dist = y_split.value_counts(normalize=True).sort_index().round(3).to_dict()
    print(f"  {split_name}: {dist}")


In [ ]:
# Escalonamento — StandardScaler (melhor para modelos lineares e distância)
scaler = StandardScaler()
X_train_sc = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_val_sc   = pd.DataFrame(scaler.transform(X_val),       columns=X_val.columns)
X_test_sc  = pd.DataFrame(scaler.transform(X_test),      columns=X_test.columns)

# Salvar scaler
joblib.dump(scaler, os.path.join(MODELS_DIR, 'scaler.joblib'))
print("✓ StandardScaler aplicado e salvo")

# Seleção de Features — Top 25 por Mutual Information
selector = SelectKBest(mutual_info_classif, k=min(25, X_train_sc.shape[1]))
X_train_sel_arr = selector.fit_transform(X_train_sc, y_train)
X_test_sel_arr  = selector.transform(X_test_sc)
X_val_sel_arr   = selector.transform(X_val_sc)

sel_mask     = selector.get_support()
sel_features = X_train_sc.columns[sel_mask].tolist()
mi_scores    = dict(zip(X_train_sc.columns, selector.scores_))

X_train_sel = pd.DataFrame(X_train_sel_arr, columns=sel_features)
X_val_sel   = pd.DataFrame(X_val_sel_arr,   columns=sel_features)
X_test_sel  = pd.DataFrame(X_test_sel_arr,  columns=sel_features)

joblib.dump(sel_features, os.path.join(MODELS_DIR, 'feature_names.joblib'))
X_test_sel.to_csv(os.path.join(DATA_PROCESSED, 'X_test.csv'), index=False)
y_test.to_csv(os.path.join(DATA_PROCESSED,  'y_test.csv'), index=False)

print(f"✓ {len(sel_features)} features selecionadas por Mutual Information")
print("\nTop 15 features selecionadas:")
top_mi = sorted(mi_scores.items(), key=lambda x: x[1], reverse=True)[:15]
for i, (feat, score) in enumerate(top_mi, 1):
    marker = '★' if feat in sel_features else ' '
    print(f"  {marker}{i:2d}. {feat:<40} MI = {score:.4f}")


## 7. Treinamento e Comparação de Modelos

In [ ]:
# Definir todos os modelos
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=42),
}

# Adicionar modelos avançados se disponíveis
try:
    models['XGBoost'] = XGBClassifier(
        n_estimators=100, random_state=42, eval_metric='mlogloss',
        use_label_encoder=False, verbosity=0
    )
except: pass
try:
    models['LightGBM'] = LGBMClassifier(n_estimators=100, random_state=42, verbose=-1)
except: pass
try:
    models['CatBoost'] = CatBoostClassifier(iterations=100, random_state=42, verbose=0)
except: pass

print(f"Modelos a treinar: {len(models)}")
for name in models: print(f"  - {name}")


In [ ]:
# Treinar e avaliar todos os modelos
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
classes = sorted(y_train.unique().tolist())
results = {}

print("\n" + "="*65)
print(f"{'MODELO':<25} {'Accuracy':>9} {'F1-Macro':>9} {'F1-Wt':>8} {'CV F1':>12}")
print("="*65)

for name, model in models.items():
    model.fit(X_train_sel, y_train)
    y_pred = model.predict(X_test_sel)
    
    cv_scores = cross_val_score(
        model, X_train_sel, y_train, cv=cv, scoring='f1_weighted', n_jobs=-1
    )
    
    y_prob = None
    try: y_prob = model.predict_proba(X_test_sel)
    except: pass
    
    roc_auc = None
    if y_prob is not None:
        try:
            y_bin = label_binarize(y_test, classes=classes)
            roc_auc = roc_auc_score(y_bin, y_prob, multi_class='ovr', average='macro')
        except: pass
    
    results[name] = {
        'model': model,
        'accuracy': accuracy_score(y_test, y_pred),
        'f1_macro': f1_score(y_test, y_pred, average='macro', zero_division=0),
        'f1_weighted': f1_score(y_test, y_pred, average='weighted', zero_division=0),
        'precision_macro': precision_score(y_test, y_pred, average='macro', zero_division=0),
        'recall_macro': recall_score(y_test, y_pred, average='macro', zero_division=0),
        'roc_auc': roc_auc,
        'cv_f1_mean': cv_scores.mean(),
        'cv_f1_std': cv_scores.std(),
        'confusion_matrix': confusion_matrix(y_test, y_pred).tolist(),
    }
    
    cv_str = f"{cv_scores.mean():.4f} ± {cv_scores.std():.4f}"
    print(f"{name:<25} {results[name]['accuracy']:>9.4f} {results[name]['f1_macro']:>9.4f} "
          f"{results[name]['f1_weighted']:>8.4f} {cv_str:>12}")

print("="*65)
best_name = max(results, key=lambda k: results[k]['f1_weighted'])
print(f"\n★ Melhor modelo: {best_name} (F1-Weighted = {results[best_name]['f1_weighted']:.4f})")


In [ ]:
# Gráfico comparativo de modelos
model_names = list(results.keys())
metrics = {
    'Accuracy':    [results[m]['accuracy']    for m in model_names],
    'F1 Macro':    [results[m]['f1_macro']    for m in model_names],
    'F1 Weighted': [results[m]['f1_weighted'] for m in model_names],
    'CV F1':       [results[m]['cv_f1_mean']  for m in model_names],
}

x = np.arange(len(model_names))
width = 0.2
fig, ax = plt.subplots(figsize=(14, 6))
colors = sns.color_palette('viridis', len(metrics))

for i, (metric_name, values) in enumerate(metrics.items()):
    offset = (i - len(metrics)/2) * width + width/2
    ax.bar(x + offset, values, width, label=metric_name, color=colors[i], alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=15, ha='right')
ax.set_ylabel('Score')
ax.set_ylim(0, 1.1)
ax.set_title('Comparação de Desempenho dos Modelos', fontsize=13, fontweight='bold')
ax.legend(loc='upper right', fontsize=9)
ax.axhline(0.5, color='red', linestyle='--', alpha=0.4, linewidth=1, label='Baseline 50%')
plt.tight_layout()
plt.show()


## 8. Otimização de Hiperparâmetros

In [ ]:
# RandomizedSearchCV para o melhor modelo
best_model_base = results[best_name]['model']

# Grade de hiperparâmetros por modelo
param_grids = {
    'Random Forest': {
        'n_estimators':    [100, 200, 300],
        'max_depth':       [None, 10, 20, 30],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf':  [1, 2, 4],
        'max_features':    ['sqrt', 'log2'],
    },
    'Gradient Boosting': {
        'n_estimators':   [100, 200],
        'learning_rate':  [0.05, 0.1, 0.2],
        'max_depth':      [3, 5, 7],
        'subsample':      [0.8, 1.0],
    },
    'XGBoost': {
        'n_estimators':   [100, 200],
        'learning_rate':  [0.05, 0.1, 0.2],
        'max_depth':      [3, 5, 7],
        'subsample':      [0.8, 1.0],
        'colsample_bytree': [0.8, 1.0],
    },
    'LightGBM': {
        'n_estimators':   [100, 200],
        'learning_rate':  [0.05, 0.1, 0.2],
        'num_leaves':     [31, 63, 127],
    },
}

if best_name in param_grids:
    cv_opt = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    search = RandomizedSearchCV(
        best_model_base,
        param_grids[best_name],
        n_iter=20,
        scoring='f1_weighted',
        cv=cv_opt,
        random_state=42,
        n_jobs=-1,
        verbose=0,
    )
    search.fit(X_train_sel, y_train)
    best_model_opt = search.best_estimator_
    print(f"✓ Otimização concluída!")
    print(f"  Melhores parâmetros: {search.best_params_}")
    print(f"  Melhor F1 CV:        {search.best_score_:.4f}")
else:
    best_model_opt = best_model_base
    print(f"  Usando modelo base (sem grid para {best_name})")

# Avaliar modelo otimizado
y_pred_opt = best_model_opt.predict(X_test_sel)
print(f"\nResultados após otimização:")
print(f"  Accuracy:    {accuracy_score(y_test, y_pred_opt):.4f}")
print(f"  F1-Macro:    {f1_score(y_test, y_pred_opt, average='macro', zero_division=0):.4f}")
print(f"  F1-Weighted: {f1_score(y_test, y_pred_opt, average='weighted', zero_division=0):.4f}")


## 9. Avaliação Detalhada do Melhor Modelo

In [ ]:
# Relatório completo de classificação
y_pred_final = best_model_opt.predict(X_test_sel)
print("RELATÓRIO DE CLASSIFICAÇÃO — MODELO OTIMIZADO")
print("=" * 55)
print(classification_report(y_test, y_pred_final,
      target_names=['Baixa(1)','Méd-Baixa(2)','Méd-Alta(3)','Alta(4)'],
      zero_division=0))


In [ ]:
# Matriz de Confusão
cm = confusion_matrix(y_test, y_pred_final)
labels = ['Sat. 1', 'Sat. 2', 'Sat. 3', 'Sat. 4']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels, ax=axes[0])
axes[0].set_title('Matriz de Confusão (Absoluta)')
axes[0].set_ylabel('Real'); axes[0].set_xlabel('Previsto')

cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues',
            xticklabels=labels, yticklabels=labels, ax=axes[1])
axes[1].set_title('Matriz de Confusão (Normalizada)')
axes[1].set_ylabel('Real'); axes[1].set_xlabel('Previsto')

fig.suptitle(f'Avaliação do Modelo — {best_name}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Curvas ROC Multi-classe
if hasattr(best_model_opt, 'predict_proba'):
    y_prob = best_model_opt.predict_proba(X_test_sel)
    y_bin  = label_binarize(y_test, classes=classes)
    
    fig, ax = plt.subplots(figsize=(9, 6))
    for i, (cls, color) in enumerate(zip(classes, SAT_COLORS)):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=color, linewidth=2.2,
                label=f'Satisfação {cls} (AUC = {roc_auc:.3f})')
    
    ax.plot([0,1],[0,1],'k--', linewidth=1, alpha=0.5)
    ax.set_xlabel('Taxa de Falso Positivo')
    ax.set_ylabel('Taxa de Verdadeiro Positivo')
    ax.set_title('Curvas ROC — Classificação Multiclasse (OvR)', fontsize=13, fontweight='bold')
    ax.legend(loc='lower right')
    ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
    plt.tight_layout()
    plt.show()


## 10. Interpretabilidade do Modelo

In [ ]:
# Feature Importance (Random Forest / Gradient Boosting / XGBoost)
if hasattr(best_model_opt, 'feature_importances_'):
    importance = pd.Series(best_model_opt.feature_importances_, index=sel_features)
    importance_top = importance.nlargest(20).sort_values()
    
    fig, ax = plt.subplots(figsize=(10, 8))
    colors = sns.color_palette('viridis', len(importance_top))
    ax.barh(importance_top.index, importance_top.values, color=colors, alpha=0.85)
    ax.axvline(importance_top.mean(), color='red', linestyle='--', alpha=0.7, label='Média')
    ax.set_xlabel('Importância (Impurity-based)')
    ax.set_title('Top 20 Features Mais Importantes', fontsize=13, fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.show()
    
    print("\nTop 10 Features:")
    for i, (feat, imp) in enumerate(importance.nlargest(10).items(), 1):
        print(f"  {i:2d}. {feat:<40} {imp:.4f}")


In [ ]:
# SHAP Values (se disponível)
try:
    import shap
    explainer = shap.TreeExplainer(best_model_opt)
    shap_values = explainer.shap_values(X_test_sel.iloc[:200])
    
    print("SHAP Values calculados!")
    
    # Summary plot (usando matplotlib diretamente)
    if isinstance(shap_values, list):
        # Multiclasse: média absoluta por classe
        shap_abs = np.mean([np.abs(sv) for sv in shap_values], axis=0)
    else:
        shap_abs = np.abs(shap_values)
    
    mean_shap = pd.Series(shap_abs.mean(axis=0), index=sel_features).nlargest(20).sort_values()
    
    fig, ax = plt.subplots(figsize=(10, 8))
    colors = sns.color_palette('viridis', len(mean_shap))
    ax.barh(mean_shap.index, mean_shap.values, color=colors, alpha=0.85)
    ax.set_xlabel('|SHAP Value| Médio')
    ax.set_title('SHAP — Importância Global das Features', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
except ImportError:
    print("⚠ SHAP não instalado. Para instalar: pip install shap")
except Exception as e:
    print(f"⚠ Erro ao calcular SHAP: {e}")


## 11. Persistência e Inferência do Modelo

In [ ]:
# Salvar modelo final
joblib.dump(best_model_opt, os.path.join(MODELS_DIR, 'best_model.joblib'))
joblib.dump(best_model_opt, os.path.join(MODELS_DIR, 'best_model.pkl'))
joblib.dump(scaler, os.path.join(MODELS_DIR, 'scaler.joblib'))
joblib.dump(sel_features, os.path.join(MODELS_DIR, 'feature_names.joblib'))

print(f"✓ Modelo salvo: models/best_model.joblib")
print(f"✓ Scaler salvo: models/scaler.joblib")
print(f"✓ Feature names: models/feature_names.joblib")
print(f"\nArquivos em models/:")
for f in os.listdir(MODELS_DIR):
    size = os.path.getsize(os.path.join(MODELS_DIR, f)) / 1024
    print(f"  {f:<35} {size:.1f} KB")


In [ ]:
# Exemplo de inferência — predição para novos funcionários
print("EXEMPLO DE INFERÊNCIA")
print("=" * 55)

# Carregar modelo
loaded_model = joblib.load(os.path.join(MODELS_DIR, 'best_model.joblib'))
loaded_scaler = joblib.load(os.path.join(MODELS_DIR, 'scaler.joblib'))
loaded_features = joblib.load(os.path.join(MODELS_DIR, 'feature_names.joblib'))

# Pegar 5 amostras do conjunto de teste
X_sample = X_test_sel.head(5)
y_real    = y_test.head(5).values
y_pred_sample = loaded_model.predict(X_sample)
y_prob_sample = loaded_model.predict_proba(X_sample)

sat_map = {1:'Baixa', 2:'Média-Baixa', 3:'Média-Alta', 4:'Alta'}
print(f"{'Func.':<8} {'Real':<14} {'Previsto':<14} {'Prob Cl.1':>9} {'Prob Cl.2':>9} {'Prob Cl.3':>9} {'Prob Cl.4':>9}")
print("-" * 75)
for i, (real, pred, prob) in enumerate(zip(y_real, y_pred_sample, y_prob_sample)):
    correct = "✓" if real == pred else "✗"
    print(f"  {i+1:<6} {sat_map.get(real,'?'):<14} {sat_map.get(pred,'?'):<14} "
          f"{prob[0]:>9.3f} {prob[1]:>9.3f} {prob[2]:>9.3f} {prob[3]:>9.3f}  {correct}")


## 12. Insights e Recomendações de Negócio

In [ ]:
# Dashboard de KPIs finais
print("=" * 65)
print("  DASHBOARD EXECUTIVO — SATISFAÇÃO DOS FUNCIONÁRIOS")
print("=" * 65)
print(f"  Total de Funcionários:        {len(df):,}")
print(f"  Satisfação Média:             {df[TARGET].mean():.2f} / 4.00")
print(f"  Funcionários Satisfeitos (≥3): {(df[TARGET]>=3).sum()} ({(df[TARGET]>=3).mean()*100:.1f}%)")
print(f"  Funcionários Insatisfeitos (≤2): {(df[TARGET]<=2).sum()} ({(df[TARGET]<=2).mean()*100:.1f}%)")
print(f"  Taxa de Attrition:            {(df['Attrition']=='Yes').mean()*100:.1f}%")
print()
print("  DESEMPENHO DO MODELO:")
print(f"  Algoritmo:                    {best_name}")
print(f"  Accuracy:                     {accuracy_score(y_test, y_pred_final):.4f}")
print(f"  F1-Macro:                     {f1_score(y_test, y_pred_final, average='macro', zero_division=0):.4f}")
print(f"  F1-Weighted:                  {f1_score(y_test, y_pred_final, average='weighted', zero_division=0):.4f}")
print("=" * 65)


In [ ]:
# Análise dos principais fatores de satisfação
print("PRINCIPAIS FATORES DE SATISFAÇÃO")
print("-" * 50)
print()

# OverTime
ot_diff = df[df['OverTime']=='No'][TARGET].mean() - df[df['OverTime']=='Yes'][TARGET].mean()
print(f"⚠ HORAS EXTRAS:")
print(f"  Funcionários SEM hora extra:  {df[df['OverTime']=='No'][TARGET].mean():.2f}")
print(f"  Funcionários COM hora extra:  {df[df['OverTime']=='Yes'][TARGET].mean():.2f}")
print(f"  Diferença:                    {ot_diff:.2f} pontos (impacto NEGATIVO)")
print()

# WorkLifeBalance
wlb_means = df.groupby('WorkLifeBalance')[TARGET].mean()
print(f"✅ WORK-LIFE BALANCE:")
for k, v in wlb_means.items():
    label = {1:'Ruim',2:'Regular',3:'Bom',4:'Ótimo'}.get(k,str(k))
    print(f"  WLB {k} ({label}): {v:.2f}")
print()

# EnvironmentSatisfaction
env_corr = df['EnvironmentSatisfaction'].corr(df[TARGET])
print(f"✅ SATISFAÇÃO COM AMBIENTE (r = {env_corr:.3f})")
print(f"  Correlação positiva com satisfação no trabalho")
print()

# Attrition por satisfação
print(f"📉 ATTRITION POR NÍVEL DE SATISFAÇÃO:")
for sat_level in sorted(df[TARGET].unique()):
    grp = df[df[TARGET]==sat_level]
    attr_rate = (grp['Attrition']=='Yes').mean()*100
    label = {1:'Baixa',2:'Média-Baixa',3:'Média-Alta',4:'Alta'}.get(sat_level)
    print(f"  Satisfação {sat_level} ({label}): {attr_rate:.1f}% de desligamentos")


## Recomendações Finais para a Empresa

### 🔴 Alta Prioridade

1. **Controle de Horas Extras** — Funcionários com overtime são significativamente menos satisfeitos. Revisar processos, redistribuir carga e considerar novas contratações em áreas sobrecarregadas.

2. **Plano de Carreira e Promoção** — Funcionários sem promoção recente apresentam declínio na satisfação. Ciclos anuais de avaliação com critérios claros e transparentes.

### 🟡 Média Prioridade

3. **Equilíbrio Vida-Trabalho** — WorkLifeBalance impacta diretamente a satisfação. Implementar trabalho remoto/híbrido e horários flexíveis.

4. **Revisão Salarial** — MonthlyIncome correlaciona com satisfação. Benchmark anual com o mercado, especialmente nos cargos de menor satisfação.

5. **Ambiente de Trabalho** — EnvironmentSatisfaction é forte preditor. Pesquisas de clima periódicas e programas de melhoria contínua.

### 🟢 Médio/Longo Prazo

6. **Programa de Retenção Proativa** — Usar o modelo para identificar funcionários em risco ANTES do pedido de demissão. Acionar gestores para conversas preventivas.

7. **Desenvolvimento e Treinamento** — Investir em capacitação, principalmente para perfis com menor satisfação.

8. **Políticas de Mobilidade** — Para funcionários com maior distância residência-trabalho: auxílio transporte, home office ou realocação.

---

> 📊 **Todos os gráficos foram salvos em `reports/figures/`**  
> 📁 **JSONs para dashboard React/Next.js em `reports/`**  
> 🤖 **Modelo treinado disponível em `models/best_model.joblib`**
